# Selene Basics: Build, Run, and Read Results

This notebook introduces the basic Selene workflow:
1. Define an input program
2. Build it with `selene_sim.build`
3. Run shots of your program
4. Inspect parsed and unparsed outputs
5. Clean up build artifacts

# 1. Define an input program

Selene's primary role is to provide a realistic yet controlable emulation environment for hybrid quantum computation. While Selene does support an interactive mode, the most common use of Selene is to run self-contained programs, just as you would on a real quantum computer.

There are several input formats that Selene supports, but most if our efforts are centred around [Guppy](https://docs.quantinuum.com/guppy/), a safe and powerful pythonic language for building hybrid quantum programs.

In this example, we allocate a fresh qubit (in the $\ket{0}$ state), perform a Hadamard gate on it to produce the $\ket{+} = \frac{1}{\sqrt{2}}\left(\ket{0} + \ket{1}\right)$ state. Then we measure it, producing 0 or 1 with equal probability, and output the measurement as "c0":

In [11]:
from guppylang import guppy
from guppylang.std.builtins import result
from guppylang.std.quantum import qubit, h, measure

@guppy
def single_qubit_demo() -> None:
    q0 = qubit()
    h(q0)
    result("c0", measure(q0))

# 2. Build the Selene instance for your program 

Selene has a bundled build system that can detect programs of different forms. In this case, we provide it with `single_qubit_demo.compile()`, which goes through a build pipeline to produce a selene instance. More information on the Selene build system is provided in a later notebook.

In [12]:
from selene_sim.build import build
runner = build(single_qubit_demo.compile())

# 3. Run simulations

Now that we have the built Selene instance of our program, we can run it and get results.

In order to run it, we require some configuration. In particular, we require at least:

- A simulator backend. We bundle some built-in simulators, and provide APIs for third parties to implement additional ones. The built-in simulators include:
  - Stim (an [open source](https://github.com/quantumlib/Stim/) stabilizer simulator, $O(N^2)$ - fast, supports only Clifford operations)
  - Quest (an [open source](https://github.com/quest-kit/QuEST) statevector simulator,  $O(2^N)$ - universal, but more computationally expensive)
  - Coinflip (random measurement outcomes, no quantum simulation is performed)
  - Classical / Quantum replay - covered in later notebooks

- The number of qubits
  - General hybrid programs are not resource-bound, but may allocate and free qubits arbitrarily, so we need to provide an upper bound (just as a real device has a maximum number of qubits).
  - Note that resources scale as the number of qubits increases, even if they are not necessarily used in the course of a program - so it's best to be conservative here.

Here we will run Selene with several basic configurations. Although we support a `run` function for single-shot runs, this will soon be deprecated and we strongly recommend using `run_shots`.

`runner.run_shots` provides an iterator over shots, and each shot is also an iterator. This allows you to see shot results in realtime as they are generated, rather than waiting for completion:

In [13]:
from selene_sim import Quest, Stim, Coinflip

n_shots = 10

quest_shots = runner.run_shots(
    simulator=Quest(),
    n_qubits=1,
    n_shots=n_shots,
    random_seed=123
)
stim_shots = runner.run_shots(
    simulator=Stim(),
    n_qubits=1,
    n_shots=n_shots,
    random_seed=123
)
coinflip_shots = runner.run_shots(
    simulator=Coinflip(),
    n_qubits=1,
    n_shots=n_shots,
    random_seed=123
)

quest_results = list(dict(shot) for shot in quest_shots)
stim_results = list(dict(shot) for shot in stim_shots)
coinflip_results = list(dict(shot) for shot in coinflip_shots)

print("Quest results   : ", ''.join(str(results['c0']) for results in quest_results))
print("Stim results    : ", ''.join(str(results['c0']) for results in stim_results))
print("Coinflip results: ", ''.join(str(results['c0']) for results in coinflip_results))

Quest results   :  0111011100
Stim results    :  0000000011
Coinflip results:  0010011010


# 4. Array outputs and unparsed mode

Selene can emit arrays and, when useful, raw tagged records (`parse_results=False`).

In [14]:
from guppylang import guppy
from guppylang.std.builtins import result, array
from guppylang.std.quantum import qubit, h, measure, x, measure_array

@guppy
def array_demo() -> None:
    qs = array(qubit() for _ in range(4))
    for i in range(4):
        if i % 2 == 0:
            x(qs[i])
    bits = measure_array(qs)
    result("bits", bits)

array_runner = build(array_demo.compile(), "array_demo")
parsed = list(array_runner.run(Quest(), n_qubits=4))
unparsed = list(array_runner.run(Quest(), n_qubits=4, parse_results=False))

print("Parsed:", parsed)
print("Unparsed:", unparsed)

Parsed: [('bits', [1, 0, 1, 0])]
Unparsed: [('USER:BOOLARR:bits', [1, 0, 1, 0])]


The unparsed interface is primarily useful for services that, in particular, wish to perform minimal processing of outputs before passing them onto a later parsing service. We will see in a later notebook how this provides an alternative interface to metrics and other useful utilities.

# 5. Build artifact cleanup

Builds produce build artifacts and run directories in your system's temporary directory by default. When performing a lot of emulation, these may build up over time, so you can opt to clean run directories:

In [15]:
print("Run dirs before cleanup:", len(list(runner.runs.iterdir())))
runner.delete_run_directories()
print("Run dirs after delete_run_directories():", len(list(runner.runs.iterdir())))

Run dirs before cleanup: 3
Run dirs after delete_run_directories(): 0


or even the full instance:

In [16]:
print("Build root exists before cleanup:", runner.root.exists())
runner.delete_files()
print("Build root exists after delete_files():", runner.root.exists())

Build root exists before cleanup: True
Build root exists after delete_files(): False


but note that removing build files will prevent the runner from working, as in the following example.

*Note: we replace the build directory with a placeholder before printing it here. This is because the build root is a random directory, and we want deterministic output - this allows us to test these outputs and ensure that any changed outputs get reviewed before pull requests are merged.*

In [17]:
try:
    list(dict(shot) for shot in runner.run_shots(Quest(), n_qubits=4, n_shots=10))
except FileNotFoundError as e:
    print(str(e).replace(str(runner.root), "<ROOT>"))

This Selene instance's root directory does not exist (<ROOT>). This is likely because the instance has been deleted. Please rebuild.
